In [1]:
!pip install -q -U transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 86.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 68.7 MB/s eta 0:00:00:00:01


In [2]:
import re
import time
import torch
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

MAX_SAMPLES = None

MAX_NEW_TOKENS = 512

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model loaded:", MODEL_NAME)

total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(f"Total parameters: {total_params / 1e6:.2f}M")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-0.6B-Base
Total parameters: 596,049,920
Total parameters: 596.05M


In [4]:
dataset = load_dataset("openai/gsm8k", "main")

test_dataset = dataset["test"]

if MAX_SAMPLES is not None:
    test_dataset = test_dataset.select(
        range(min(MAX_SAMPLES, len(test_dataset)))
    )

print("Number of test examples:", len(test_dataset))

print("\nExample question:")
print(test_dataset[0]["question"])

print("\nGold solution:")
print(test_dataset[0]["answer"])

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Number of test examples: 1319

Example question:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Gold solution:
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


In [5]:
def create_prompt(question):
    return f"""Solve the following math problem step by step.

Question:
{question}

At the end, write the final numerical answer exactly in this format:

Final Answer: <number>

Answer:
"""

In [6]:
def extract_gold_answer(answer_text):
    """
    GSM8K answers contain:
    #### 42
    """
    match = re.search(r"####\s*([-+]?[0-9,]*\.?[0-9]+)", answer_text)

    if match:
        return match.group(1).replace(",", "")

    return None


def extract_model_answer(text):
    match = re.search(
        r"Final Answer:\s*([-+]?[0-9,]*\.?[0-9]+)",
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1).replace(",", "")

    return None

In [7]:
print(
    extract_gold_answer(
        test_dataset[0]["answer"]
    )
)

18


In [8]:
question = test_dataset[0]["question"]

prompt = create_prompt(question)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

gold_answer = extract_gold_answer(
    test_dataset[0]["answer"]
)

predicted_answer = extract_model_answer(response)

print("QUESTION")
print(question)

print("\nMODEL RESPONSE")
print(response)

print("\nPREDICTED ANSWER:", predicted_answer)
print("GOLD ANSWER:", gold_answer)

print(
    "CORRECT:",
    predicted_answer == gold_answer
)

QUESTION
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

MODEL RESPONSE
Step 1: Calculate the number of eggs Janet eats for breakfast every morning.
Eggs eaten for breakfast = 3 eggs

Step 2: Calculate the number of eggs Janet bakes for her friends every day.
Eggs baked for friends = 4 eggs

Step 3: Calculate the total number of eggs Janet lays per day.
Total eggs per day = 16 eggs

Step 4: Calculate the number of eggs Janet sells at the farmers' market daily.
Eggs sold at the farmers' market = Total eggs per day - Eggs eaten for breakfast - Eggs baked for friends
Eggs sold at the farmers' market = 16 - 3 - 4 = 9 eggs

Step 5: Calculate the total amount of money Janet makes every day at the farmers' market.
Money made at the farmers' market = Eggs sold at 

In [9]:
import time
import torch
import pandas as pd
from tqdm.auto import tqdm

BATCH_SIZE = 32
MAX_NEW_TOKENS = 512

# Important for batched generation with decoder-only models
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


results = []
correct = 0
valid_count = 0

start_time = time.time()

for start_idx in tqdm(
    range(0, len(test_dataset), BATCH_SIZE),
    desc="Evaluating"
):
    end_idx = min(start_idx + BATCH_SIZE, len(test_dataset))

    batch = test_dataset.select(range(start_idx, end_idx))

    questions = batch["question"]
    gold_solutions = batch["answer"]

    prompts = [
        create_prompt(question)
        for question in questions
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove the prompt tokens
    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]

    responses = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    for question, gold_solution, response in zip(
        questions,
        gold_solutions,
        responses
    ):
        gold_answer = extract_gold_answer(gold_solution)
        predicted_answer = extract_model_answer(response)

        valid_format = predicted_answer is not None
        is_correct = predicted_answer == gold_answer

        if valid_format:
            valid_count += 1

        if is_correct:
            correct += 1

        results.append({
            "question": question,
            "gold_answer": gold_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct,
            "valid_format": valid_format,
            "response": response
        })


elapsed_time = time.time() - start_time

accuracy = correct / len(test_dataset)
valid_format_rate = valid_count / len(test_dataset)

print("\n================================")
print("ZERO-SHOT BASELINE")
print("================================")
print("Model:", MODEL_NAME)
print("Samples:", len(test_dataset))
print("Batch size:", BATCH_SIZE)

print(f"Correct: {correct}/{len(test_dataset)}")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Valid format rate: {valid_format_rate * 100:.2f}%")

print(f"Evaluation time: {elapsed_time / 60:.2f} minutes")
print(
    f"Average time/question: "
    f"{elapsed_time / len(test_dataset):.3f} seconds"
)

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]


ZERO-SHOT BASELINE
Model: Qwen/Qwen3-0.6B-Base
Samples: 1319
Batch size: 32
Correct: 648/1319
Accuracy: 49.13%
Valid format rate: 83.09%
Evaluation time: 22.30 minutes
Average time/question: 1.014 seconds


In [10]:
results_df = pd.DataFrame(results)

results_df.head()

,question,gold_answer,predicted_answer,correct,valid_format,response
0,Janet’s ducks lay 16 eggs per day. She eats th...,18,18,True,True,Step 1: Calculate the number of eggs Janet eat...
1,A robe takes 2 bolts of blue fiber and half th...,3,3,True,True,Step 1: Determine the amount of white fiber ne...
2,Josh decides to try flipping a house. He buys...,70000,120000,False,True,Step 1: Calculate the increased value of the h...
3,James decides to run 3 sprints 3 times a week....,540,None,False,False,Step 1: Calculate the total distance James run...
4,"Every day, Wendi feeds each of her chickens th...",20,-20,False,True,Step 1: Calculate the total amount of feed giv...


In [11]:
incorrect_df = results_df[
    results_df["correct"] == False
]

print("Incorrect examples:", len(incorrect_df))

incorrect_df[
    [
        "question",
        "gold_answer",
        "predicted_answer",
        "response"
    ]
].head(10)

Incorrect examples: 671


,question,gold_answer,predicted_answer,response
2,Josh decides to try flipping a house. He buys...,70000,120000,Step 1: Calculate the increased value of the h...
3,James decides to run 3 sprints 3 times a week....,540,None,Step 1: Calculate the total distance James run...
4,"Every day, Wendi feeds each of her chickens th...",20,-20,Step 1: Calculate the total amount of feed giv...
7,Carla is downloading a 200 GB file. Normally s...,160,100,Step 1: Calculate the time it takes to downloa...
8,John drives for 3 hours at a speed of 60 mph a...,45,60,Step 1: Calculate the distance John drives in ...
12,Carlos is planting a lemon tree. The tree will...,13,7,Step 1: Calculate the profit per year\nProfit ...
13,Melanie is a door-to-door saleswoman. She sold...,18,8,Step 1: Let's denote the total number of vacuu...
15,A merchant wants to make a choice of purchase ...,125,0,Step 1: Calculate the increase in the value of...
19,Marissa is hiking a 12-mile trail. She took 1 ...,6,4,Step 1: Calculate the total time Marissa has a...
20,I have 10 liters of orange drink that are two-...,15,None,Step 1: Calculate the amount of water in the o...


In [12]:
results_df.to_csv(
    "qwen3_0.6b_base_gsm8k_zero_shot.csv",
    index=False
)

summary = {
    "experiment": "zero_shot",
    "model": MODEL_NAME,
    "dataset": "GSM8K",
    "num_samples": len(test_dataset),
    "correct": correct,
    "accuracy": accuracy,
    "total_parameters": total_params,
    "trainable_parameters": 0,
    "evaluation_seconds": elapsed_time,
}

summary_df = pd.DataFrame([summary])

summary_df.to_csv(
    "zero_shot_summary.csv",
    index=False
)

summary_df

,experiment,model,dataset,num_samples,correct,accuracy,total_parameters,trainable_parameters,evaluation_seconds
0,zero_shot,Qwen/Qwen3-0.6B-Base,GSM8K,1319,648,0.491281,596049920,0,1337.845901
